In [6]:
import random
import copy
from statistics import mean

# -------------------------------
# Config / Tunables
# -------------------------------
GAME_MINUTES = 48
SECONDS_PER_MIN = 60
GAME_SECONDS = GAME_MINUTES * SECONDS_PER_MIN
QUARTER_SECONDS = 12 * SECONDS_PER_MIN

DEFAULT_POSSESSIONS_GOAL = None  # If None, we simulate by time; otherwise simulate N possessions per team roughly.

# Sub rotations
SUB_CHECK_EVERY_POSSESSIONS = 4  # check subs roughly every N possessions
FATIGUE_PLAY_PENALTY = 0.006     # multiplier to reduce shot% per fatigue unit
FATIGUE_REB_PENALTY = 0.03       # reduce reb chance per fatigue unit (affects reb weights)
FATIGUE_DEF_PENALTY = 0.004      # reduces defense impact vs shooter per fatigue

# Coach AI thresholds
COACH_AGGRESSIVE_DEFICIT = 8     # if down by this many points and enough time -> go aggressive
COACH_AGGRESSIVE_MINUTES_LEFT = 6

# Play type probabilities baseline
PLAYTYPE_BASE = {
    'transition': 0.10,
    'pnr': 0.30,
    'iso': 0.20,
    'spotup': 0.25,
    'post': 0.15
}

# Team average references for analytics targets (for tuning)
NBA_PACE_TARGET = 99
NBA_OFFRTG_TARGET = 114

# -------------------------------
# Helper: Create Player Template
# -------------------------------
def create_player(name, role='wing'):
    # roles: 'pg','wing','big' roughly influence usage and 3pt tendency
    role = role
    base_2p = random.uniform(0.47, 0.56) if role in ('pg','wing') else random.uniform(0.50, 0.60)
    base_3p = random.uniform(0.33, 0.40) if role != 'big' else random.uniform(0.20, 0.30)
    usage = random.uniform(0.8, 1.4) if role == 'pg' else random.uniform(0.7, 1.3)
    return {
        'name': name,
        'role': role,
        '2P%': base_2p,
        '3P%': base_3p,
        'FT%': random.uniform(0.72, 0.91),
        'usage': usage,
        'TO%': random.uniform(0.09, 0.16),
        'clutch_boost': random.uniform(0.01, 0.04),
        'fatigue': 0.0,
        'fouls': 0,
        'disqualified': False,
        'sit_until': 0,
        'ast_per_game': random.uniform(1.5, 8.0),
        'orb_per_game': random.uniform(0.5, 3.5),
        'drb_per_game': random.uniform(1.5, 8.0),
        'stats': {
            'points': 0, 'fouls': 0, 'possessions': 0,
            'assists': 0, 'off_reb': 0, 'def_reb': 0,
            'tech_fouls': 0, 'flagrant_fouls': 0,
            'fgm': 0, 'fga': 0, '3pm': 0, '3pa': 0, 'ftm': 0, 'fta': 0
        }
    }

# -------------------------------
# Utility: swap players (substitution)
# -------------------------------
def swap_players(on_court, bench, idx_on, bench_player):
    # swap on_court[idx_on] with bench_player (which must be in bench)
    player_out = on_court[idx_on]
    if bench_player not in bench:
        return False
    bench.remove(bench_player)
    bench.append(player_out)
    on_court[idx_on] = bench_player
    return True

# -------------------------------
# Substitution Logic (improved)
# -------------------------------
def perform_subs(team_starters, team_bench, team_name, game_seconds_left, quarter_seconds,
                 current_quarter, score_diff, coach_style='balanced', log=None):
    """
    Checks bench for subs based on fatigue, fouls, and coach AI signals.
    coach_style in {'balanced','aggressive','conservative'} affects subs frequency.
    """
    # thresholds tuned to role
    subs_made = 0
    # attempt to sub anyone with foul trouble or high fatigue
    for i, p in enumerate(list(team_starters)):
        if p.get('disqualified', False) or p['fouls'] >= 6:
            # immediate sub
            eligible = [b for b in team_bench if not b.get('disqualified', False) and b.get('sit_until',0) <= game_seconds_left]
            if eligible:
                sub = max(eligible, key=lambda x: x['usage'])  # pick best available
                swap_players(team_starters, team_bench, i, sub)
                sub['fatigue'] = max(0, sub['fatigue'] - 3)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fouled out: {p['name']} -> {sub['name']}")
        elif p['fatigue'] > 25 or (p['fatigue'] > 18 and coach_style == 'balanced'):
            # sub due to fatigue
            eligible = [b for b in team_bench if not b.get('disqualified', False) and b.get('sit_until',0) <= game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x: x['fatigue'])  # freshest
                swap_players(team_starters, team_bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub for fatigue: {p['name']} -> {sub['name']}")
        # situational subs by coach
        elif coach_style == 'conservative' and score_diff > 10 and game_seconds_left > 5*60:
            # rest starters if ahead big
            eligible = [b for b in team_bench if b.get('sit_until',0) <= game_seconds_left]
            if eligible:
                sub = min(eligible, key=lambda x: x['fatigue'])
                swap_players(team_starters, team_bench, i, sub)
                subs_made += 1
                if log: log.append(f"{team_name} - Sub to rest (leading): {p['name']} -> {sub['name']}")
    return subs_made

# -------------------------------
# Rebounding Logic (improved weights + fatigue)
# -------------------------------
def get_rebound(off_court, def_court, is_offensive, log, team_name):
    if is_offensive:
        reb_weights = [max(0.01, p['orb_per_game'] - p['fatigue'] * FATIGUE_REB_PENALTY) for p in off_court]
        rebounder = random.choices(off_court, weights=reb_weights, k=1)[0]
        rebounder['stats']['off_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed an offensive rebound!")
    else:
        reb_weights = [max(0.01, p['drb_per_game'] - p['fatigue'] * FATIGUE_REB_PENALTY) for p in def_court]
        rebounder = random.choices(def_court, weights=reb_weights, k=1)[0]
        rebounder['stats']['def_reb'] += 1
        log.append(f"{team_name} - {rebounder['name']} grabbed a defensive rebound!")
    return rebounder

# -------------------------------
# Shot and Play Type Helpers
# -------------------------------
def pick_play_type(team, opponent, is_transition, game_seconds_left, score_diff, coach_aggressive=False):
    # baseline probabilities
    probs = PLAYTYPE_BASE.copy()
    if is_transition:
        probs = {k: v * (1.5 if k == 'transition' else 0.5) for k, v in probs.items()}
    # coach adjustments: if down and enough time, favor quick plays and 3-pt
    if coach_aggressive and game_seconds_left > COACH_AGGRESSIVE_MINUTES_LEFT * 60 and score_diff < -COACH_AGGRESSIVE_DEFICIT:
        probs['transition'] += 0.05
        probs['iso'] += 0.05
        probs['spotup'] += 0.03
    # normalize
    total = sum(probs.values())
    for k in probs: probs[k] /= total
    choices, weights = zip(*probs.items())
    return random.choices(choices, weights=weights, k=1)[0]

def shot_modifier_by_play(play_type):
    # influences shot selection & efficiency
    if play_type == 'transition':
        return {'2P_mult': 1.05, '3P_mult': 0.9, 'rebound_effort': 0.9}
    if play_type == 'pnr':
        return {'2P_mult': 1.02, '3P_mult': 1.03, 'rebound_effort': 1.0}
    if play_type == 'iso':
        return {'2P_mult': 0.95, '3P_mult': 1.05, 'rebound_effort': 0.95}
    if play_type == 'spotup':
        return {'2P_mult': 0.95, '3P_mult': 1.10, 'rebound_effort': 0.9}
    if play_type == 'post':
        return {'2P_mult': 1.10, '3P_mult': 0.7, 'rebound_effort': 1.1}
    return {'2P_mult': 1.0, '3P_mult': 1.0, 'rebound_effort': 1.0}

# -------------------------------
# Simulate a Possession (full)
# -------------------------------
def simulate_possession(off_court, def_court, game_seconds_left, team_fouls, is_home, log,
                        offense_team_name, defense_team_name, team_def_rating, opponent_def_rating,
                        coach_aggressive=False):
    # decide transition by a small chance or via last event (passed in externally could be improved)
    is_transition = random.random() < 0.08  # 8% of possessions are transition by default
    shooter = random.choices(off_court, weights=[p['usage'] for p in off_court], k=1)[0]
    # play type
    play_type = pick_play_type(off_court, def_court, is_transition, game_seconds_left, 
                               score_diff=team_fouls.get('score_diff',0), coach_aggressive=coach_aggressive)
    play_mod = shot_modifier_by_play(play_type)

    # fatigue penalties (affects accuracy and defense)
    fatigue_penalty = min(0.25, shooter['fatigue'] * FATIGUE_PLAY_PENALTY)  # capped
    fatigue_def_pen = min(0.25, shooter['fatigue'] * FATIGUE_DEF_PENALTY)

    clutch_bonus = shooter.get('clutch_boost', 0.0) if game_seconds_left <= 120 else 0.0
    home_bonus = 0.02 if is_home else 0.0

    # team defense scales: higher defensive rating reduces shooter's chance
    # team_def_rating is e.g., 1.0 baseline; values <1 are better defense (less opponent FG%)
    team_def_factor = team_def_rating  # multiplier applied to shooter's effective % (e.g., 0.98 reduces 2%)
    opp_def_factor = opponent_def_rating

    # dynamic shot selection: favor 3 if shooter and game context supports it
    # compute shooter's 3pt propensity
    shooter_3pt_skill = shooter['3P%']
    # context: if down and late, more 3s
    if game_seconds_left < 2*60 and team_fouls.get('score_diff',0) < 0:
        # trailing in last 2 minutes -> more 3s
        three_bias = 0.25
    else:
        three_bias = 0.0

    # build outcome weights: 2P, 3P, TO, Foul
    # base weights influenced by play type and shooter tendencies
    base_two_weight = 0.50 * play_mod['2P_mult']
    base_three_weight = (0.20 + (0.5 * shooter_3pt_skill)) * play_mod['3P_mult']  # more skilled shooters attempt more 3s
    if shooter['role'] == 'big':
        base_three_weight *= 0.5
    turnover_weight = shooter['TO%'] * 1.0
    foul_weight = 0.14

    # tune by context
    base_three_weight += three_bias
    # ensure minimums and renormalize
    weights = [max(0.01, base_two_weight), max(0.01, base_three_weight), max(0.01, turnover_weight), max(0.01, foul_weight)]
    outcome = random.choices(['2P', '3P', 'TO', 'Foul'], weights=weights, k=1)[0]

    # possession duration roughly depends on play type and coach decisions
    base_possession_time = {
        'transition': random.randint(5, 10),
        'pnr': random.randint(8, 18),
        'iso': random.randint(7, 18),
        'spotup': random.randint(6, 16),
        'post': random.randint(8, 20)
    }.get(play_type, random.randint(8, 16))

    # coach may alter pace: if aggressive, slightly shorter possessions
    if coach_aggressive:
        base_possession_time = max(4, int(base_possession_time * 0.9))

    possession_time = base_possession_time
    points = 0
    rebound_team = None

    if outcome in ['2P', '3P']:
        base_prob = shooter['2P%'] if outcome == '2P' else shooter['3P%']
        # apply modifiers: fatigue, play type, defense, home, clutch
        effective_prob = base_prob * play_mod['2P_mult' if outcome == '2P' else '3P_mult']
        effective_prob = effective_prob - fatigue_penalty - (team_def_factor - 1.0) * 0.02 + clutch_bonus + home_bonus
        effective_prob = max(0.03, min(0.95, effective_prob))
        made = random.random() < effective_prob
        # update attempts
        shooter['stats']['fga'] += 1
        if outcome == '3P':
            shooter['stats']['3pa'] += 1
        if made:
            pts = 2 if outcome == '2P' else 3
            points += pts
            shooter['stats']['fgm'] += 1
            if outcome == '3P':
                shooter['stats']['3pm'] += 1
            # update shooter points
            shooter['stats']['points'] += pts
            # possible assist
            eligible_passers = [p for p in off_court if p['name'] != shooter['name']]
            if eligible_passers and random.random() < 0.28:  # baseline assist chance on made field goal
                assister = random.choices(eligible_passers, weights=[p['ast_per_game'] for p in eligible_passers], k=1)[0]
                assister['stats']['assists'] += 1
                log.append(f"{offense_team_name} - {assister['name']} assisted {shooter['name']} ({play_type})")
        else:
            # missed shot -> rebound contested
            off_reb_prob = 0.28 * play_mod.get('rebound_effort', 1.0)
            # defender fatigue reduces their reb effectiveness
            is_offensive_reb = random.random() < off_reb_prob
            rebound_team = off_court if is_offensive_reb else def_court
            get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
            possession_time += random.randint(2, 5)

    elif outcome == 'TO':
        # turnover: no shot, defense gets possession
        # chance for steal (defensive stat) could be added
        rebound_team = def_court
        # small chance of live-ball turnover leading to transition
        possession_time = random.randint(3, 8)

    elif outcome == 'Foul':
        defender = random.choice(def_court)
        defender['fouls'] += 1
        defender['stats']['fouls'] = defender['fouls']

        # determine foul type
        foul_type = random.choices(['regular', 'shooting', 'technical', 'flagrant'],
                                   weights=[0.75, 0.18, 0.05, 0.02], k=1)[0]

        # team fouls counting
        count_toward = foul_type in ('regular', 'shooting')
        if count_toward:
            team_fouls[defense_team_name] += 1

        # probability base for FT
        prob_base = min(0.98, max(0.5, shooter['FT%'] - fatigue_penalty + clutch_bonus + home_bonus))
        if foul_type == 'technical':
            # one free throw (team keeps possession)
            made = random.random() < prob_base
            shooter['stats']['ftm'] += int(made)
            shooter['stats']['fta'] += 1
            shooter['stats']['points'] += int(made)
            points += int(made)
            defender['stats']['tech_fouls'] += 1
            rebound_team = off_court
        elif foul_type == 'flagrant':
            # two free throws + possession
            fts = [random.random() < prob_base for _ in range(2)]
            made = sum(1 for m in fts if m)
            shooter['stats']['fta'] += 2
            shooter['stats']['ftm'] += made
            shooter['stats']['points'] += made
            points += made
            defender['stats']['flagrant_fouls'] += 1
            rebound_team = off_court
        elif foul_type == 'shooting':
            # shooting foul: depending on shot location, choose attempts
            shot_type = random.choices(['2P', '3P'], weights=[0.8, 0.2], k=1)[0]
            if random.random() < 0.25:
                # "and-1": made shot + 1 FT
                base_made_pts = 2 if shot_type == '2P' else 3
                shooter['stats']['points'] += base_made_pts
                shooter['stats']['fgm'] += 1
                shooter['stats']['fga'] += 1
                if shot_type == '3P':
                    shooter['stats']['3pm'] += 1
                    shooter['stats']['3pa'] += 1
                # one free throw
                made_ft = random.random() < prob_base
                shooter['stats']['fta'] += 1
                shooter['stats']['ftm'] += int(made_ft)
                shooter['stats']['points'] += int(made_ft)
                points += base_made_pts + int(made_ft)
            else:
                # missed -> 2 or 3 FTs
                ft_attempts = 2 if shot_type == '2P' else 3
                fts = [random.random() < prob_base for _ in range(ft_attempts)]
                made = sum(1 for m in fts if m)
                shooter['stats']['fta'] += ft_attempts
                shooter['stats']['ftm'] += made
                shooter['stats']['points'] += made
                points += made
                # if miss on any FT, chance for rebound
                if any(not m for m in fts):
                    off_reb_prob = 0.32
                    is_offensive_reb = random.random() < off_reb_prob
                    rebound_team = off_court if is_offensive_reb else def_court
                    get_rebound(off_court, def_court, is_offensive_reb, log, offense_team_name if is_offensive_reb else defense_team_name)
                    possession_time += 2
        else:
            # regular non-shooting foul -> ball stays with offense or free throws if bonus
            bonus = team_fouls[defense_team_name] >= 5
            if bonus:
                fts = [random.random() < prob_base for _ in range(2)]
                made = sum(1 for m in fts if m)
                shooter['stats']['fta'] += 2
                shooter['stats']['ftm'] += made
                shooter['stats']['points'] += made
                points += made
            else:
                # possession continues but shorter
                possession_time = random.randint(3, 7)
                rebound_team = off_court

        # foul out?
        if defender['fouls'] >= 6:
            defender['disqualified'] = True
            log.append(f"{defense_team_name} - {defender['name']} has fouled out!")

    # finalize possession bookkeeping
    shooter['stats']['possessions'] += 1
    shooter['fatigue'] += 1 + (2 if play_type == 'transition' else 0)  # transition more tiring
    # defense players fatigue a little
    for p in def_court:
        p['fatigue'] += 0.2

    return points, possession_time, rebound_team, play_type

# -------------------------------
# Game Simulation
# -------------------------------
def simulate_game(home_team_name, home_roster, away_team_name, away_roster,
                  home_def_rating=1.0, away_def_rating=1.0,
                  coach_home='balanced', coach_away='balanced', verbose=False):
    """
    home_def_rating / away_def_rating: multiplies into effective shot success (1.0 = baseline; <1 better D)
    returns detailed stats and log
    """
    # copies
    home = copy.deepcopy(home_roster)
    away = copy.deepcopy(away_roster)
    # starters first 5
    home_starters, home_bench = home[:5], home[5:]
    away_starters, away_bench = away[:5], away[5:]

    # reset stats
    for p in home + away:
        p['fatigue'] = 0.0
        p['fouls'] = 0
        p['disqualified'] = False
        p['sit_until'] = 0
        p['stats'] = {'points': 0, 'fouls': 0, 'possessions': 0,
                      'assists': 0, 'off_reb': 0, 'def_reb': 0,
                      'tech_fouls': 0, 'flagrant_fouls': 0,
                      'fgm': 0, 'fga': 0, '3pm': 0, '3pa': 0, 'ftm': 0, 'fta': 0}

    # game flow
    score = {home_team_name: 0, away_team_name: 0}
    team_fouls = {home_team_name: 0, away_team_name: 0}
    possession_team = home_team_name
    time_left = GAME_SECONDS
    log = []
    possessions = 0
    # count possessions per team to compute pace
    team_possessions = {home_team_name: 0, away_team_name: 0}

    # coach settings
    coach_state = {home_team_name: {'style': coach_home, 'aggressive': False},
                   away_team_name: {'style': coach_away, 'aggressive': False}}

    # main loop by time
    while time_left > 0:
        # update coach aggression if trailing late
        for tname in (home_team_name, away_team_name):
            other = away_team_name if tname == home_team_name else home_team_name
            lead = score[tname] - score[other]
            # if trailing and time is limited -> go aggressive
            if lead < -COACH_AGGRESSIVE_DEFICIT and time_left <= COACH_AGGRESSIVE_MINUTES_LEFT * 60:
                coach_state[tname]['aggressive'] = True
            else:
                coach_state[tname]['aggressive'] = False

        # set offense/defense arrays
        if possession_team == home_team_name:
            off_court = home_starters
            def_court = away_starters
            is_home = True
            off_def_rating = away_def_rating
            def_def_rating = home_def_rating
            offense_name = home_team_name
            defense_name = away_team_name
        else:
            off_court = away_starters
            def_court = home_starters
            is_home = False
            off_def_rating = home_def_rating
            def_def_rating = away_def_rating
            offense_name = away_team_name
            defense_name = home_team_name

        # provide score_diff via team_fouls dict for play selection
        team_fouls['score_diff'] = score[offense_name] - score[defense_name]

        pts, dur, rebound_team, play_type = simulate_possession(
            off_court, def_court, game_seconds_left=time_left, team_fouls=team_fouls,
            is_home=is_home, log=log, offense_team_name=offense_name, defense_team_name=defense_name,
            team_def_rating=(off_def_rating), opponent_def_rating=(def_def_rating),
            coach_aggressive=coach_state[offense_name]['aggressive']
        )

        # apply points to scoreboard
        score[possession_team] += pts
        # also increment team possessions
        team_possessions[possession_team] += 1
        possessions += 1

        # advance time by possession duration (approx)
        time_left -= dur
        if time_left < 0: time_left = 0

        # substitutions periodic check
        if possessions % SUB_CHECK_EVERY_POSSESSIONS == 0:
            # compute score_diff for subs logic relative to each team
            home_diff = score[home_team_name] - score[away_team_name]
            away_diff = -home_diff
            perform_subs(home_starters, home_bench, home_team_name, time_left, QUARTER_SECONDS, None, home_diff, coach_style=coach_state[home_team_name]['style'], log=log)
            perform_subs(away_starters, away_bench, away_team_name, time_left, QUARTER_SECONDS, None, away_diff, coach_style=coach_state[away_team_name]['style'], log=log)

        # possession change logic:
        # If rebound_team is defensive team list (def_court) or None -> change possession
        if rebound_team is None:
            # No offensive rebound -> change possession
            possession_team = away_team_name if possession_team == home_team_name else home_team_name
        else:
            # if rebound_team == off_court then same team keeps possession, else change
            if rebound_team == def_court:
                possession_team = away_team_name if possession_team == home_team_name else home_team_name
            else:
                # offense retains; no change
                pass

        # reset team fouls at quarter break
        elapsed = GAME_SECONDS - time_left
        # when time left equals exact multiple -> end of quarter (be careful with integer)
        if time_left % QUARTER_SECONDS == 0 and time_left != GAME_SECONDS:
            # reset fouls
            team_fouls = {home_team_name: 0, away_team_name: 0}
            # relieve some fatigue for resting at quarter
            for p in home_starters + away_starters + home_bench + away_bench:
                p['fatigue'] = max(0, p['fatigue'] - 2)
                p['sit_until'] = 0
            log.append(f"--- End of Q{int(elapsed//QUARTER_SECONDS)} - Fouls reset and small rest ---")

    # end of game
    # compile analytics
    def compile_team_stats(team_name, starters, bench):
        players = starters + bench
        pts = sum(p['stats']['points'] for p in players)
        fgm = sum(p['stats'].get('fgm', 0) for p in players)
        fga = sum(p['stats'].get('fga', 0) for p in players)
        threem = sum(p['stats'].get('3pm', 0) for p in players)
        threea = sum(p['stats'].get('3pa', 0) for p in players)
        ftm = sum(p['stats'].get('ftm', 0) for p in players)
        fta = sum(p['stats'].get('fta', 0) for p in players)
        ast = sum(p['stats']['assists'] for p in players)
        orb = sum(p['stats']['off_reb'] for p in players)
        drb = sum(p['stats']['def_reb'] for p in players)
        possessions_count = team_possessions[team_name] if team_name in team_possessions else 0
        fg_pct = fgm / fga if fga > 0 else 0
        three_pct = threem / threea if threea > 0 else 0
        efg = (fgm + 0.5 * threem) / fga if fga > 0 else 0
        off_rtg = (pts / possessions_count * 100) if possessions_count > 0 else 0
        return {
            'pts': pts, 'fgm': fgm, 'fga': fga, 'fg%': fg_pct,
            '3pm': threem, '3pa': threea, '3p%': three_pct,
            'ftm': ftm, 'fta': fta, 'ast': ast, 'orb': orb, 'drb': drb,
            'eFG%': efg, 'OffRtg': off_rtg, 'possessions': possessions_count
        }

    home_stats = compile_team_stats(home_team_name, home_starters, home_bench)
    away_stats = compile_team_stats(away_team_name, away_starters, away_bench)

    # Pace estimation
    total_poss = team_possessions[home_team_name] + team_possessions[away_team_name]
    pace = total_poss * (48 / GAME_MINUTES) if GAME_MINUTES > 0 else 0

    # player lists
    players_final = {p['name']: p['stats'] for p in home + away}

    # optional verbose logs truncated
    if verbose:
        print(f"FINAL SCORE: {home_team_name} {score[home_team_name]} - {away_team_name} {score[away_team_name]}")
        print(f"Home team stats: {home_stats}")
        print(f"Away team stats: {away_stats}")
        print(f"Pace estimate: {pace:.1f}, Total possessions: {total_poss}")

    return {
        'score': score,
        'home_stats': home_stats,
        'away_stats': away_stats,
        'players': players_final,
        'log': log,
        'pace': pace,
        'team_possessions': team_possessions
    }

# -------------------------------
# Monte Carlo wrapper (optional)
# -------------------------------
def monte_carlo(home_team_name, away_team_name, home_roster, away_roster, n=10, verbose=False):
    results = []
    for i in range(n):
        res = simulate_game(home_team_name, home_roster, away_team_name, away_roster,
                            home_def_rating=random.uniform(0.96,1.04),
                            away_def_rating=random.uniform(0.96,1.04),
                            coach_home=random.choice(['balanced','aggressive','conservative']),
                            coach_away=random.choice(['balanced','aggressive','conservative']),
                            verbose=False)
        results.append(res)
    # aggregate team totals
    def agg(key):
        return mean([r[key]['pts'] if key in r else 0 for r in results]) if isinstance(results[0][key], dict) else None

    # quick summary
    if verbose:
        for i, r in enumerate(results):
            print(f"Game {i+1}: {r['score'][home_team_name]} - {r['score'][away_team_name]}; Pace {r['pace']:.1f}")
    return results



# -------------------------------
# Single Game Simulation Wrapper
# -------------------------------
def simulate_single_game(home_team_name, away_team_name, home_roster, away_roster,
                         home_def_rating=1.0, away_def_rating=1.0,
                         coach_home='balanced', coach_away='balanced'):
    """
    Runs a single detailed simulation with possession-by-possession log
    and full box score with advanced statistics.
    """
    print(f"=== Simulating {home_team_name} vs {away_team_name} (Single Game Mode) ===\n")
    result = simulate_game(
        home_team_name, home_roster, away_team_name, away_roster,
        home_def_rating=home_def_rating,
        away_def_rating=away_def_rating,
        coach_home=coach_home,
        coach_away=coach_away,
        verbose=False
    )

    # Print every possession (play-by-play)
    print("\n--- PLAY-BY-PLAY LOG ---")
    for line in result["log"]:
        print(line)
    print("\n--- END OF GAME ---")

    # Print game summary
    print(f"\nFINAL SCORE: {home_team_name} {result['score'][home_team_name]} - {away_team_name} {result['score'][away_team_name]}")
    print(f"Pace: {result['pace']:.1f}  |  Possessions: {result['team_possessions']}")
    print("\nTEAM STATS:")
    print(f"{home_team_name} - {result['home_stats']}")
    print(f"{away_team_name} - {result['away_stats']}")

    # Box Score
    print("\n--- BOX SCORE ---")
    all_players = result['players']
    for name, stats in sorted(all_players.items(), key=lambda kv: kv[1].get('points', 0), reverse=True):
        print(f"{name:10s} | {stats['points']:2d} pts | FGM/FGA {stats['fgm']}/{stats['fga']} | "
              f"3P {stats['3pm']}/{stats['3pa']} | FT {stats['ftm']}/{stats['fta']} | AST {stats['assists']} | "
              f"REB {stats['off_reb']+stats['def_reb']} ({stats['off_reb']}/{stats['def_reb']}) | "
              f"Fouls {stats['fouls']}")

    # Top players
    top_scorers = sorted(all_players.items(), key=lambda kv: kv[1].get('points', 0), reverse=True)[:5]
    print("\nTop Scorers:")
    for name, stats in top_scorers:
        print(f" - {name}: {stats['points']} pts, {stats['fgm']}/{stats['fga']} FG, {stats['3pm']}/{stats['3pa']} 3P")

    print("\nAdvanced Stats:")
    for team, stats in zip([home_team_name, away_team_name], [result['home_stats'], result['away_stats']]):
        print(f"{team}: eFG% {stats['eFG%']:.3f}, OffRtg {stats['OffRtg']:.1f}")

    return result


# -------------------------------
# Multi-Game Simulation Wrapper
# -------------------------------
def simulate_multiple_games(home_team_name, away_team_name, home_roster, away_roster,
                            n=50, verbose=False):
    """
    Runs many games without play-by-play logs, aggregates full box scores and analytics.
    """
    print(f"=== Simulating {n} games between {home_team_name} and {away_team_name} (Batch Mode) ===")
    results = monte_carlo(home_team_name, away_team_name, home_roster, away_roster, n=n, verbose=False)

    # Aggregate team metrics
    home_pts = [r['score'][home_team_name] for r in results]
    away_pts = [r['score'][away_team_name] for r in results]
    home_wins = sum(1 for h, a in zip(home_pts, away_pts) if h > a)
    away_wins = n - home_wins
    avg_pace = mean(r['pace'] for r in results)

    def avg_team_stats(key):
        return {
            'pts': mean(r[key]['pts'] for r in results),
            'fg%': mean(r[key]['fg%'] for r in results),
            '3p%': mean(r[key]['3p%'] for r in results),
            'eFG%': mean(r[key]['eFG%'] for r in results),
            'OffRtg': mean(r[key]['OffRtg'] for r in results),
        }

    home_avg = avg_team_stats('home_stats')
    away_avg = avg_team_stats('away_stats')

    print("\n--- SERIES RESULTS ---")
    print(f"{home_team_name}: {home_wins}-{away_wins} record over {n} games")
    print(f"Average Pace: {avg_pace:.1f}\n")

    print(f"{home_team_name} Avg Stats: {home_avg}")
    print(f"{away_team_name} Avg Stats: {away_avg}")

    return {
        'home_avg': home_avg,
        'away_avg': away_avg,
        'pace': avg_pace,
        'home_wins': home_wins,
        'away_wins': away_wins
    }


# -------------------------------
# Example Usage
# -------------------------------
if __name__ == "__main__":
    # build rosters with roles
    home_roster = [create_player(f"LAL_{r}", role=random.choice(['pg','wing','big'])) for r in ["PG","W1","W2","B1","B2","BEN1","BEN2","BEN3","BEN4","BEN5"]]
    away_roster = [create_player(f"BOS_{r}", role=random.choice(['pg','wing','big'])) for r in ["PG","W1","W2","B1","B2","BEN1","BEN2","BEN3","BEN4","BEN5"]]

    # Single detailed game
    result = simulate_single_game("LAL", "BOS", home_roster, away_roster,
                                  home_def_rating=0.98, away_def_rating=1.02,
                                  coach_home='balanced', coach_away='aggressive')

    # Multi-game Monte Carlo run
    print("\n\n--- MULTI-GAME ANALYSIS ---")
    series = simulate_multiple_games("LAL", "BOS", home_roster, away_roster, n=20)





=== Simulating LAL vs BOS (Single Game Mode) ===


--- PLAY-BY-PLAY LOG ---
LAL - LAL_W1 grabbed a defensive rebound!
LAL - LAL_W2 grabbed a defensive rebound!
LAL - LAL_B2 grabbed a defensive rebound!
LAL - LAL_PG assisted LAL_B1 (post)
LAL - LAL_W1 grabbed a defensive rebound!
LAL - LAL_W2 grabbed an offensive rebound!
LAL - LAL_W2 assisted LAL_B2 (pnr)
BOS - BOS_PG grabbed an offensive rebound!
LAL - LAL_B2 grabbed an offensive rebound!
BOS - BOS_W1 grabbed a defensive rebound!
BOS - BOS_W1 grabbed a defensive rebound!
BOS - BOS_B1 grabbed a defensive rebound!
LAL - LAL_PG assisted LAL_W2 (pnr)
BOS - BOS_B1 grabbed a defensive rebound!
BOS - BOS_W1 assisted BOS_B1 (iso)
BOS - BOS_PG grabbed a defensive rebound!
LAL - LAL_B2 grabbed a defensive rebound!
BOS - BOS_PG grabbed a defensive rebound!
LAL - LAL_W2 grabbed an offensive rebound!
LAL - LAL_B2 grabbed a defensive rebound!
LAL - LAL_W1 grabbed a defensive rebound!
BOS - BOS_PG grabbed a defensive rebound!
BOS - BOS_B2 grabbed a 